In [33]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, matthews_corrcoef, classification_report, confusion_matrix

In [14]:
train_rf_df = pd.read_csv('/content/drive/MyDrive/diplomado_cm_ia/model_ensemble/data/train_selected_RandomForest.csv')
test_rf_df = pd.read_csv('/content/drive/MyDrive/diplomado_cm_ia/model_ensemble/data/test_selected_RandomForest.csv')


y_train_rf = train_rf_df['Class']
x_train_rf = train_rf_df.drop('Class', axis=1)

y_test_rf = test_rf_df['Class']
x_test_rf = test_rf_df.drop('Class', axis=1)

train_knn_df = pd.read_csv('/content/drive/MyDrive/diplomado_cm_ia/model_ensemble/data/train_selected_knn.csv')
test_knn_df = pd.read_csv('/content/drive/MyDrive/diplomado_cm_ia/model_ensemble/data/test_selected_knn.csv')

y_train_knn = train_knn_df['Class']
x_train_knn = train_knn_df.drop('Class', axis=1)

y_test_knn = test_knn_df['Class']
x_test_knn = test_knn_df.drop('Class', axis=1)



In [37]:
def build_rf_and_predict():
  model = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')
  model.fit(x_train_rf, y_train_rf)

  print("\n Modelo Random Forest entrenado exitosamente.")
  # Realizar predicciones en el conjunto de prueba
  y_pred = model.predict(x_test_rf)

  # Evaluar el rendimiento del modelo
  # mcc = matthews_corrcoef(y_test_rf, y_pred)
  # print(f"MCC del modelo Random Forest: {mcc:.4f}")

  # print("\nReporte de Clasificación:\n")
  # print(classification_report(y_test_rf, y_pred))

  # print("\nMatriz de Confusión:\n")
  # print(confusion_matrix(y_test_rf, y_pred))

  return {'y_pred': y_pred, 'mcc':matthews_corrcoef(y_test_rf, y_pred), 'acc':accuracy_score(y_test_rf, y_pred)}


In [38]:
def build_knn_and_predict():
  model = KNeighborsClassifier(n_neighbors=5)
  model.fit(x_train_knn, y_train_knn)

  print("\n Modelo KNN entrenado exitosamente.")
  # Realizar predicciones en el conjunto de prueba
  y_pred = model.predict(x_test_knn)

  # Evaluar el rendimiento del modelo
  # mcc = matthews_corrcoef(y_test_knn, y_pred)
  # print(f"MCC del modelo KNN: {mcc:.4f}")

  # print("\nReporte de Clasificación:\n")
  # print(classification_report(y_test_knn, y_pred))

  # print("\nMatriz de Confusión:\n")
  # print(confusion_matrix(y_test_knn, y_pred))

  return {'y_pred': y_pred, 'mcc':matthews_corrcoef(y_test_knn, y_pred), 'acc':accuracy_score(y_test_knn, y_pred)}

In [39]:
def diversity_measures(y_true, y_pred1, y_pred2):

    y_true = np.array(y_true)
    y_pred1 = np.array(y_pred1)
    y_pred2 = np.array(y_pred2)

    correct1 = (y_pred1 == y_true)
    correct2 = (y_pred2 == y_true)

    N11 = np.sum(correct1 & correct2)
    N00 = np.sum(~correct1 & ~correct2)
    N10 = np.sum(correct1 & ~correct2)
    N01 = np.sum(~correct1 & correct2)

    N = len(y_true)

    disagreement = (N10 + N01)
    double_fault = N00

    return {
        "N11": N11,
        "N00": N00,
        "N10": N10,
        "N01": N01,
        "disagreement": disagreement,
        "double_fault": double_fault
    }

In [41]:
rf_results = build_rf_and_predict()
svc_results = build_knn_and_predict()

div_rf_knn = diversity_measures(y_test_rf, rf_results['y_pred'], svc_results['y_pred'])

df_div = pd.DataFrame([
    {
        'model': 'RF_KNN',
        "disagreement": div_rf_knn['disagreement'],
        "double_fault": div_rf_knn['double_fault']
    }
])

print('\n Diversidad \n')
print(df_div)

print('\n Metricas \n')
df_metrics = pd.DataFrame([
    {
        'model': 'RandomForest',
        'mcc': rf_results['mcc'],
        'acc': rf_results['acc']
    },
    {
        'model': 'SVC',
        'mcc': svc_results['mcc'],
        'acc': svc_results['acc']
    }
])

print(df_metrics)




 Modelo Random Forest entrenado exitosamente.

 Modelo KNN entrenado exitosamente.

 Diversidad 

    model  disagreement  double_fault
0  RF_KNN            15             6

 Metricas 

          model       mcc       acc
0  RandomForest  0.583383  0.790323
1           SVC  0.548387  0.774194


In [46]:
y_ensemble = np.logical_or(rf_results['y_pred'], svc_results['y_pred'])

print(classification_report(y_test_rf, y_ensemble))
print(matthews_corrcoef(y_test_rf, y_ensemble))

              precision    recall  f1-score   support

         APP       0.76      0.84      0.80        31
      nonAPP       0.82      0.74      0.78        31

    accuracy                           0.79        62
   macro avg       0.79      0.79      0.79        62
weighted avg       0.79      0.79      0.79        62

0.583383351196948
